# Train SFW-SwinCBM on Google Colab
Notebook duy nhat de train tu data tren Google Drive, luu checkpoint/result ve Drive, va co progress bar theo epoch.

## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())


## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Clone repo branch ai_core

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/chtr302/ai_generated_image_detection.git'
BRANCH = 'ai_core'
REPO_DIR = Path('/content/ai_generated_image_detection')

os.chdir('/content')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--single-branch', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('cwd:', Path.cwd())
print('branch:', subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip())
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)


## 4. Install dependencies

In [ ]:
import os
from pathlib import Path

REPO_DIR = Path('/content/ai_generated_image_detection')
os.chdir(REPO_DIR)
!python -m pip install -q datasets pyarrow pillow tqdm gdown


## 5. Verify repo version

In [ ]:
import subprocess
import sys

train_help = subprocess.run([sys.executable, '-m', 'src.model.train', '--help'], check=True, capture_output=True, text=True).stdout
benchmark_help = subprocess.run([sys.executable, '-m', 'src.model.benchmark', '--help'], check=True, capture_output=True, text=True).stdout
required = ['--data-root', '--max-train-steps', '--max-val-steps', '--log-every', '--progress']
missing = [arg for arg in required if arg not in train_help]
missing += [arg for arg in ['--data-root', '--max-samples'] if arg not in benchmark_help]
if missing:
    subprocess.run(['git', 'log', '-1', '--oneline'], check=False)
    raise RuntimeError('Repo tren Colab dang la code cu, thieu args: ' + ', '.join(missing))
print('OK: repo supports this notebook')


## 6. Prepare data from Drive
Cell nay tai folder Drive ve local Colab, sau do tao folder anh `train/val/test/real/ai` de train bang `--data-root`. Neu folder Drive khong public, set `DRIVE_DATA_DIR` thanh duong dan trong MyDrive.

In [ ]:
import shutil
from pathlib import Path

from datasets import load_dataset
from PIL import Image
from tqdm.auto import tqdm

SOURCE_FOLDER_URL = 'https://drive.google.com/drive/folders/1rWtzTG8VYBaMnuYFTB6ZZCkFrNvzH_83?usp=sharing'
DRIVE_DATA_DIR = None  # Example: Path('/content/drive/MyDrive/ai_data/defactify')
RAW_DATA_DIR = Path('/content/drive_source_data')
DATA_ROOT = Path('/content/ai_detector_images')
OUTPUT_DIR = Path('/content/drive/MyDrive/ai_detector_outputs')
REFRESH_DATA = False
EXPORT_IMAGE_SIZE = 256

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def _download_or_copy_source() -> Path:
    if REFRESH_DATA and RAW_DATA_DIR.exists():
        shutil.rmtree(RAW_DATA_DIR)
    if RAW_DATA_DIR.exists() and any(RAW_DATA_DIR.rglob('*.parquet')):
        return RAW_DATA_DIR

    if DRIVE_DATA_DIR is not None:
        source = Path(DRIVE_DATA_DIR)
        if not source.exists():
            raise FileNotFoundError(f'DRIVE_DATA_DIR khong ton tai: {source}')
        RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
        for file in source.rglob('*'):
            if file.is_file():
                target = RAW_DATA_DIR / file.relative_to(source)
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(file, target)
    else:
        import gdown
        if RAW_DATA_DIR.exists():
            shutil.rmtree(RAW_DATA_DIR)
        RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
        gdown.download_folder(url=SOURCE_FOLDER_URL, output=str(RAW_DATA_DIR), quiet=False, use_cookies=False, remaining_ok=True)
    return RAW_DATA_DIR


def _infer_data_files(root: Path) -> dict[str, list[str]]:
    files = sorted(root.rglob('*.parquet'))
    if not files:
        raise FileNotFoundError('Khong tim thay file data trong folder Drive. Kiem tra share link hoac DRIVE_DATA_DIR.')

    split_files = {'train': [], 'validation': [], 'test': []}
    for file in files:
        name = file.name.lower()
        parts = {part.lower() for part in file.relative_to(root).parts[:-1]}
        if 'train' in name or 'train' in parts:
            split_files['train'].append(str(file))
        elif 'validation' in name or 'valid' in name or 'val' in name or {'validation', 'valid', 'val'} & parts:
            split_files['validation'].append(str(file))
        elif 'test' in name or 'test' in parts:
            split_files['test'].append(str(file))

    result = {split: paths for split, paths in split_files.items() if paths}
    if 'train' not in result:
        raise RuntimeError('Khong tu nhan dien duoc train split. Hay dat ten file/folder co chu train/validation/test.')
    return result


def _to_rgb_image(value):
    if isinstance(value, Image.Image):
        return value.convert('RGB')
    if isinstance(value, dict):
        if value.get('bytes') is not None:
            from io import BytesIO
            return Image.open(BytesIO(value['bytes'])).convert('RGB')
        if value.get('path') is not None:
            return Image.open(value['path']).convert('RGB')
    if isinstance(value, (str, Path)):
        return Image.open(value).convert('RGB')
    if hasattr(value, '__array__'):
        return Image.fromarray(value).convert('RGB')
    raise TypeError(f'Khong doc duoc image field: {type(value)!r}')


def _export_split(dataset, split_name: str) -> int:
    out_split = 'val' if split_name == 'validation' else split_name
    count = 0
    for index, row in enumerate(tqdm(dataset, desc=f'export {out_split}', unit='img')):
        label = int(row['Label_A'])
        label_name = 'ai' if label == 1 else 'real'
        out_dir = DATA_ROOT / out_split / label_name
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f'{index:08d}.jpg'
        if not out_path.exists():
            image = _to_rgb_image(row['Image'])
            image.thumbnail((EXPORT_IMAGE_SIZE, EXPORT_IMAGE_SIZE), Image.Resampling.BICUBIC)
            image.save(out_path, format='JPEG', quality=95)
        count += 1
    return count


def prepare_data_root() -> Path:
    if REFRESH_DATA and DATA_ROOT.exists():
        shutil.rmtree(DATA_ROOT)
    if DATA_ROOT.exists() and any(DATA_ROOT.rglob('*.jpg')):
        print('Using existing DATA_ROOT:', DATA_ROOT)
        return DATA_ROOT

    raw_root = _download_or_copy_source()
    data_files = _infer_data_files(raw_root)
    print('Detected data files:', {key: len(value) for key, value in data_files.items()})
    ds = load_dataset('parquet', data_files=data_files)
    counts = {split: _export_split(ds[split], split) for split in ds.keys()}
    print('Exported:', counts)
    return DATA_ROOT


DATA_ROOT = prepare_data_root()
TRAIN_IMAGE_COUNT = len(list((DATA_ROOT / 'train').rglob('*.jpg')))
print('DATA_ROOT:', DATA_ROOT)
print('TRAIN_IMAGE_COUNT:', TRAIN_IMAGE_COUNT)
print('OUTPUT_DIR:', OUTPUT_DIR)


def build_data_arg():
    return f'--data-root {DATA_ROOT}'


## 7. Train config overnight
Mac dinh uu tien chay on dinh qua dem tren T4. Neu OOM, giam `BATCH_SIZE` ve 32. Neu con du VRAM, thu 128.

In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 10
TRAIN_STEPS_PER_EPOCH = (TRAIN_IMAGE_COUNT + BATCH_SIZE - 1) // BATCH_SIZE

TRAIN_CFG = {
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'grad_accum': 1,
    'nec': 10,
    'amp': 'fp16',
    'num_workers': 2,
    'max_train_steps': TRAIN_STEPS_PER_EPOCH,
    'max_val_steps': 2,
    'log_every': 20,
    'progress': 'bar',
}

print('TRAIN_STEPS_PER_EPOCH:', TRAIN_STEPS_PER_EPOCH)
print('TRAIN_CFG:', TRAIN_CFG)


def build_train_cli_args(cfg, resume=None):
    cli_parts = [
        build_data_arg(),
        f"--output-dir {OUTPUT_DIR}",
        f"--image-size {cfg['image_size']}",
        f"--batch-size {cfg['batch_size']}",
        f"--epochs {cfg['epochs']}",
        f"--grad-accum {cfg['grad_accum']}",
        f"--nec {cfg['nec']}",
        f"--amp {cfg['amp']}",
        f"--num-workers {cfg['num_workers']}",
        f"--max-train-steps {cfg['max_train_steps']}",
        f"--max-val-steps {cfg['max_val_steps']}",
        f"--log-every {cfg['log_every']}",
        f"--progress {cfg['progress']}",
    ]
    if resume is not None:
        cli_parts.append(f"--resume {resume}")
    return ' '.join(cli_parts)


## 8. Train overnight

In [ ]:
RESUME_FROM_LAST = False
RESUME_PATH = OUTPUT_DIR / 'last.pt'
resume = RESUME_PATH if RESUME_FROM_LAST and RESUME_PATH.exists() else None
train_args = build_train_cli_args(TRAIN_CFG, resume=resume)
print('resume:', resume)
!python -m src.model.train {train_args}


## 9. Benchmark test 100 images

In [ ]:
BENCHMARK_SAMPLES = 100
CHECKPOINT = OUTPUT_DIR / 'best.pt'
COMPARE_JSON = None

if not CHECKPOINT.exists():
    raise FileNotFoundError(f'Khong tim thay checkpoint best.pt: {CHECKPOINT}')

benchmark_args = ' '.join([
    build_data_arg(),
    f"--checkpoint {CHECKPOINT}",
    "--split test",
    f"--max-samples {BENCHMARK_SAMPLES}",
    f"--image-size {IMAGE_SIZE}",
    f"--batch-size {BATCH_SIZE}",
    f"--num-workers {TRAIN_CFG['num_workers']}",
    f"--nec {TRAIN_CFG['nec']}",
    f"--amp {TRAIN_CFG['amp']}",
])
if COMPARE_JSON is not None:
    benchmark_args += f" --compare-json {COMPARE_JSON}"

!python -m src.model.benchmark {benchmark_args}


## 10. Test one image

In [ ]:
IMAGE_PATH = Path('/content/drive/MyDrive/sample.jpg')
CHECKPOINT = OUTPUT_DIR / 'best.pt'

if not IMAGE_PATH.exists():
    raise FileNotFoundError(f'Khong tim thay anh test: {IMAGE_PATH}')
if not CHECKPOINT.exists():
    raise FileNotFoundError(f'Khong tim thay checkpoint best.pt: {CHECKPOINT}')

!python -m src.model.inference --image {IMAGE_PATH} --checkpoint {CHECKPOINT} --image-size {IMAGE_SIZE} --nec {TRAIN_CFG['nec']}
